In [13]:
import pandas as pd

In [14]:
# Load dataset
df = pd.read_csv("../preparation/teammetrics.csv")
print(df.shape)
print(df.columns)

(16578, 48)
Index(['GAME_DATE', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'TEAM_NAME', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE',
       'OPP', 'WIN', 'HOME', 'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS',
       'WIN_STREAK', 'LOSE_STREAK', 'DAYS_REST', 'IS_BACK_TO_BACK',
       'SEASON_WIN_PCT', 'ELO', 'LEAGUE_POSITION', 'ROLL3_PTS',
       'ROLL3_PLUS_MINUS', 'ROLL3_FG_PCT', 'ROLL3_REB', 'ROLL3_AST',
       'ROLL3_TOV'],
      dtype='str')


In [15]:
def add_opponent_columns(df, cols, id_cols=("GAME_ID", "TEAM_ABBREVIATION")):
    """
    Brings forward a set of columns as their opponent's version.
    
    df       : the main dataframe (must have an 'OPP' column)
    cols     : list of column names to bring forward as OPP_<col>
    id_cols  : (game key, team key) used to build and merge the lookup
    """
    game_key, team_key = id_cols
    
    lookup = df[[game_key, team_key] + cols].copy()
    
    rename_map = {team_key: "OPP"}
    rename_map.update({c: f"OPP_{c}" for c in cols})
    lookup = lookup.rename(columns=rename_map)
    
    df = pd.merge(df, lookup, on=[game_key, "OPP"], how="left")
    return df

In [16]:
cols_to_bring_forward = [
    'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS', 'WIN_STREAK',
    'LOSE_STREAK', 'DAYS_REST', 'IS_BACK_TO_BACK', 'SEASON_WIN_PCT', 'ELO',
    'LEAGUE_POSITION', 'ROLL3_PTS', 'ROLL3_PLUS_MINUS', 'ROLL3_FG_PCT', 'ROLL3_REB',
    'ROLL3_AST', 'ROLL3_TOV'
]

df = add_opponent_columns(df, cols_to_bring_forward)

In [17]:
def add_diff_features(df, base_cols, drop_originals=False):
    missing = [c for c in base_cols if c not in df.columns or f"OPP_{c}" not in df.columns]
    if missing:
        raise ValueError(f"Missing base or OPP_ column for: {missing}")
    
    for col in base_cols:
        df[f"{col}_DIFF"] = df[col] - df[f"OPP_{col}"]
    
    if drop_originals:
        cols_to_drop = base_cols + [f"OPP_{c}" for c in base_cols]
        df = df.drop(columns=cols_to_drop)
    
    return df

In [18]:
diff_cols = [
    "ELO",
    "SEASON_WIN_PCT",
    "WIN_STREAK",
    "LEAGUE_POSITION",
    "ROLL3_PLUS_MINUS",
    "ROLL3_FG_PCT",
    "ROLL3_PTS",
    "ROLL3_REB",
    "PREV_PLUSMINUS",
    "PREV_PTS",
]

df = add_diff_features(df, diff_cols, drop_originals=False)

In [19]:
# Sort date
df = df.sort_values(by='GAME_DATE', ascending=True)

In [20]:
print(df.columns)

Index(['GAME_DATE', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'TEAM_NAME', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE',
       'OPP', 'WIN', 'HOME', 'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS',
       'WIN_STREAK', 'LOSE_STREAK', 'DAYS_REST', 'IS_BACK_TO_BACK',
       'SEASON_WIN_PCT', 'ELO', 'LEAGUE_POSITION', 'ROLL3_PTS',
       'ROLL3_PLUS_MINUS', 'ROLL3_FG_PCT', 'ROLL3_REB', 'ROLL3_AST',
       'ROLL3_TOV', 'OPP_PREV_WIN', 'OPP_PREV_PTS', 'OPP_PREV_PLUSMINUS',
       'OPP_WIN_STREAK', 'OPP_LOSE_STREAK', 'OPP_DAYS_REST',
       'OPP_IS_BACK_TO_BACK', 'OPP_SEASON_WIN_PCT', 'OPP_ELO',
       'OPP_LEAGUE_POSITION', 'OPP_ROLL3_PTS', 'OPP_ROLL3_PLUS_MINUS',
       'OPP_ROLL3_FG_PCT', 'OPP_ROLL3_REB', 'OPP_ROLL3_AST', 'OPP_ROLL3_TOV',
       'ELO_DIFF', 'SEASON_WIN_PCT_DIFF', 'WIN_STREAK_DIFF',
       'LEAGUE

In [21]:
# Save to csv
df.to_csv("../preparation/opponentvalues.csv", index=False)
print(df.shape)

(16578, 74)
